In [26]:
import pandas as pd 
from datetime import datetime

In [27]:
customer = pd.read_csv('datasets/olist_customers_dataset.csv')
order_items = pd.read_csv('datasets/olist_order_items_dataset.csv')
order_payments = pd.read_csv('datasets/olist_order_payments_dataset.csv')
orders = pd.read_csv('datasets/olist_orders_dataset.csv')
order_products = pd.read_csv('datasets/olist_products_dataset.csv')
geolocation = pd.read_csv('datasets/olist_geolocation_dataset.csv')
sellers = pd.read_csv('datasets/olist_sellers_dataset.csv')

In [28]:
product_translation = pd.read_csv("datasets/product_category_name_translation.csv")
product_translation.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [29]:
order_products['product_name'] = order_products['product_category_name'].map(product_translation.set_index('product_category_name')['product_category_name_english'])
order_products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_name
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares


In [30]:
order_products.rename(columns={'product_name_lenght': 'product_name_length','product_description_lenght': 'product_description_length'}, inplace=True)

In [31]:
order_products['product_name'].dropna()
order_products['product_name_length'].fillna(0)
order_products['product_description_length'].fillna(0)
order_products.head()

,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_name
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares


In [32]:
order_small = orders[['order_id','order_status','customer_id']]
order_items_small = order_items[['order_id','product_id','price','freight_value']]
order_products_small = order_products[['product_id','product_name','product_name_length','product_description_length','product_photos_qty']]

In [33]:
final_orders = order_small.merge(order_items_small,on='order_id')\
                .merge(order_products_small,on='product_id')
final_orders.head()

,order_id,order_status,customer_id,product_id,price,freight_value,product_name,product_name_length,product_description_length,product_photos_qty
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,9ef432eb6251297304e76186b10a928d,87285b34884572647811a353c7ac498a,29.99,8.72,housewares,40.0,268.0,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,b0830fb4747a6c6d20dea0b8c802d7ef,595fac2a385ac33a80bd5114aec74eb8,118.70,22.76,perfumery,29.0,178.0,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,41ce2a54c0b03bf3443c3d931a367089,aa4383b373c6aca5d8797843e5594415,159.90,19.22,auto,46.0,232.0,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,f88197465ea7920adcdbec7375364d82,d0b61bfb1de832b15ba9d266ca96e5b0,45.00,27.20,pet_shop,59.0,468.0,3.0
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,8ab97904e6daea8866dbdbc4fb7aad2c,65266b2da20d04dbe00c5c2d3bb7859e,19.90,8.72,stationery,38.0,316.0,4.0


In [34]:
final_orders.isnull().sum()

order_id                         0
order_status                     0
customer_id                      0
product_id                       0
price                            0
freight_value                    0
product_name                  1627
product_name_length           1603
product_description_length    1603
product_photos_qty            1603
dtype: int64

In [35]:
final_orders = final_orders.dropna(subset=['product_name'])

In [36]:
final_orders.isnull().sum()

order_id                      0
order_status                  0
customer_id                   0
product_id                    0
price                         0
freight_value                 0
product_name                  0
product_name_length           0
product_description_length    0
product_photos_qty            0
dtype: int64

In [37]:
final_orders = final_orders.merge(customer[['customer_id', 'customer_city']], on='customer_id', how='left')
final_orders.head()

,order_id,order_status,customer_id,product_id,price,freight_value,product_name,product_name_length,product_description_length,product_photos_qty,customer_city
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,9ef432eb6251297304e76186b10a928d,87285b34884572647811a353c7ac498a,29.99,8.72,housewares,40.0,268.0,4.0,sao paulo
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,b0830fb4747a6c6d20dea0b8c802d7ef,595fac2a385ac33a80bd5114aec74eb8,118.70,22.76,perfumery,29.0,178.0,1.0,barreiras
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,41ce2a54c0b03bf3443c3d931a367089,aa4383b373c6aca5d8797843e5594415,159.90,19.22,auto,46.0,232.0,1.0,vianopolis
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,f88197465ea7920adcdbec7375364d82,d0b61bfb1de832b15ba9d266ca96e5b0,45.00,27.20,pet_shop,59.0,468.0,3.0,sao goncalo do amarante
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,8ab97904e6daea8866dbdbc4fb7aad2c,65266b2da20d04dbe00c5c2d3bb7859e,19.90,8.72,stationery,38.0,316.0,4.0,santo andre


In [38]:
final_orders = final_orders.merge(orders[['order_estimated_delivery_date', 'order_delivered_customer_date', 'order_id']], on='order_id', how='left')

In [39]:
order_reviews = pd.read_csv('datasets/olist_order_reviews_dataset.csv')
order_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [40]:
final_orders = final_orders.merge(order_reviews[['order_id','review_score']],on='order_id')

In [41]:
final_orders.head()

,order_id,order_status,customer_id,product_id,price,freight_value,product_name,product_name_length,product_description_length,product_photos_qty,customer_city,order_estimated_delivery_date,order_delivered_customer_date,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,9ef432eb6251297304e76186b10a928d,87285b34884572647811a353c7ac498a,29.99,8.72,housewares,40.0,268.0,4.0,sao paulo,2017-10-18 00:00:00,2017-10-10 21:25:13,4
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,b0830fb4747a6c6d20dea0b8c802d7ef,595fac2a385ac33a80bd5114aec74eb8,118.70,22.76,perfumery,29.0,178.0,1.0,barreiras,2018-08-13 00:00:00,2018-08-07 15:27:45,4
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,41ce2a54c0b03bf3443c3d931a367089,aa4383b373c6aca5d8797843e5594415,159.90,19.22,auto,46.0,232.0,1.0,vianopolis,2018-09-04 00:00:00,2018-08-17 18:06:29,5
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,f88197465ea7920adcdbec7375364d82,d0b61bfb1de832b15ba9d266ca96e5b0,45.00,27.20,pet_shop,59.0,468.0,3.0,sao goncalo do amarante,2017-12-15 00:00:00,2017-12-02 00:28:42,5
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,8ab97904e6daea8866dbdbc4fb7aad2c,65266b2da20d04dbe00c5c2d3bb7859e,19.90,8.72,stationery,38.0,316.0,4.0,santo andre,2018-02-26 00:00:00,2018-02-16 18:17:02,5


In [42]:
final_orders.isna().sum()

order_id                            0
order_status                        0
customer_id                         0
product_id                          0
price                               0
freight_value                       0
product_name                        0
product_name_length                 0
product_description_length          0
product_photos_qty                  0
customer_city                       0
order_estimated_delivery_date       0
order_delivered_customer_date    2293
review_score                        0
dtype: int64

In [43]:
final_orders.rename(columns = {'order_estimated_delivery_date':'estimated_delivery_date','order_delivered_customer_date':'actual_delivery_date'},inplace = True)
final_orders.head()

,order_id,order_status,customer_id,product_id,price,freight_value,product_name,product_name_length,product_description_length,product_photos_qty,customer_city,estimated_delivery_date,actual_delivery_date,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,9ef432eb6251297304e76186b10a928d,87285b34884572647811a353c7ac498a,29.99,8.72,housewares,40.0,268.0,4.0,sao paulo,2017-10-18 00:00:00,2017-10-10 21:25:13,4
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,b0830fb4747a6c6d20dea0b8c802d7ef,595fac2a385ac33a80bd5114aec74eb8,118.70,22.76,perfumery,29.0,178.0,1.0,barreiras,2018-08-13 00:00:00,2018-08-07 15:27:45,4
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,41ce2a54c0b03bf3443c3d931a367089,aa4383b373c6aca5d8797843e5594415,159.90,19.22,auto,46.0,232.0,1.0,vianopolis,2018-09-04 00:00:00,2018-08-17 18:06:29,5
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,f88197465ea7920adcdbec7375364d82,d0b61bfb1de832b15ba9d266ca96e5b0,45.00,27.20,pet_shop,59.0,468.0,3.0,sao goncalo do amarante,2017-12-15 00:00:00,2017-12-02 00:28:42,5
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,8ab97904e6daea8866dbdbc4fb7aad2c,65266b2da20d04dbe00c5c2d3bb7859e,19.90,8.72,stationery,38.0,316.0,4.0,santo andre,2018-02-26 00:00:00,2018-02-16 18:17:02,5


In [44]:
final_orders['actual_delivery_date'] = pd.to_datetime(final_orders['actual_delivery_date'],errors = 'coerce')
final_orders['estimated_delivery_date'] = pd.to_datetime(final_orders['estimated_delivery_date'],errors = 'coerce')

final_orders['is_delayed'] = final_orders['actual_delivery_date'] > final_orders['estimated_delivery_date']
final_orders.dtypes

order_id                              object
order_status                          object
customer_id                           object
product_id                            object
price                                float64
freight_value                        float64
product_name                          object
product_name_length                  float64
product_description_length           float64
product_photos_qty                   float64
customer_city                         object
estimated_delivery_date       datetime64[ns]
actual_delivery_date          datetime64[ns]
review_score                           int64
is_delayed                              bool
dtype: object

In [45]:
final_orders.to_csv('datasets/preprocessed_dataset.csv',index=False)